# 🔬 NeuroDiff — Interactive Exploration Notebook

Run this notebook cell-by-cell to explore the diffusion model internals.  
Works on **CPU** (slow but functional) or **GPU** (fast).  
No labelled dataset required.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/NeuroDiff/blob/main/notebooks/explore.ipynb)

In [ ]:
# ── 0. Setup (Colab only) ─────────────────────────────────────────────────
# Colab opens this .ipynb directly from GitHub — it does NOT clone the repo,
# so src/ does not exist on the VM yet. We clone it here before importing.
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/YOUR_USERNAME/NeuroDiff.git'  # <-- update this
    if not os.path.exists('/content/NeuroDiff'):
        !git clone -q {REPO_URL} /content/NeuroDiff
    %cd /content/NeuroDiff/notebooks
    !pip install -q diffusers transformers accelerate shap gradio opencv-python-headless

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────
import sys, os

# Make the project root (one level up from notebooks/) importable.
# Works whether the kernel's cwd is notebooks/ (local Jupyter) or
# /content/NeuroDiff/notebooks (Colab, after the clone above).
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.model import NeuroDiffModel
from src.attribution import ConceptAttributor
from src import visualize as viz

print(f'Project root: {project_root}')
print('All imports OK ✓')

In [ ]:
# ── 2. Load model ─────────────────────────────────────────────────────────
# First run downloads ~200 MB from HuggingFace Hub
model = NeuroDiffModel()

In [ ]:
# ── 3. Generate an image ──────────────────────────────────────────────────
result = model.generate(
    num_inference_steps=50,
    seed=42,
    capture_every=5,
)

print(f"Captured {len(result['latent_frames'])} frames")
print(f"Final image shape: {result['final_image'].shape}")

In [ ]:
# ── 4. View the denoising strip ───────────────────────────────────────────
os.makedirs('../outputs', exist_ok=True)
strip_path = viz.denoising_strip(
    result['latent_frames'],
    result['attention_snapshots'],
    '../outputs/strip_notebook.png',
)

plt.figure(figsize=(16, 5))
plt.imshow(np.array(Image.open(strip_path)))
plt.axis('off')
plt.title('Denoising progression + attention maps', color='white', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Explore a specific timestep ───────────────────────────────────────
STEP_IDX = 5   # change this to explore different steps

frame = result['latent_frames'][STEP_IDX]
snap  = result['attention_snapshots'][STEP_IDX]

overlay = viz.apply_heatmap(frame['image'], snap['map'], alpha=0.55)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
fig.patch.set_facecolor('#111')
for ax, img, title in zip(
    axes,
    [frame['image'], snap['map'], overlay],
    [f"Step {frame['step']} (t={frame['timestep']})", 'Attention map', 'Overlay'],
):
    ax.imshow(img, interpolation='nearest')
    ax.set_title(title, color='#ccc', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Counterfactual generation ─────────────────────────────────────────
cf_result = model.counterfactual_generate(
    seed=42,
    edit_timestep_start=10,
    edit_timestep_end=30,
    noise_scale=0.4,
    num_inference_steps=50,
)

comp_path = viz.counterfactual_comparison(
    result['latent_frames'],
    cf_result['frames'],
    cf_result['edit_window'],
    '../outputs/cf_notebook.png',
)

plt.figure(figsize=(18, 7))
plt.imshow(np.array(Image.open(comp_path)))
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. SHAP attribution ───────────────────────────────────────────────────
# Note: runs ~30s on CPU with n_samples=64
attributor = ConceptAttributor(grid_size=4, n_samples=64)
shap_map, segments = attributor.compute(result['final_image'])

from PIL import Image as PILImage
big = np.array(PILImage.fromarray(result['final_image']).resize((128, 128), PILImage.NEAREST))
big_shap = np.array(
    PILImage.fromarray((shap_map * 255).astype(np.uint8)).resize((128, 128), PILImage.NEAREST)
) / 255.0
blended = viz.apply_heatmap(big, big_shap, cmap=viz.SHAP_CMAP, alpha=0.55)

panel_path = viz.shap_panel(big, big_shap, blended, '../outputs/shap_notebook.png')

plt.figure(figsize=(10, 4))
plt.imshow(np.array(Image.open(panel_path)))
plt.axis('off')
plt.tight_layout()
plt.show()

top = attributor.top_segments(shap_map, segments, top_k=4)
print(f'Top 4 influential segments: {top}')

In [ ]:
# ── 8. Export denoising GIF ───────────────────────────────────────────────
gif_path = viz.export_gif(
    [f['image'] for f in result['latent_frames']],
    '../outputs/denoising_notebook.gif',
    duration_ms=150,
    upscale=8,
)
print(f'GIF saved to {gif_path}')

# Display inline (Jupyter)
from IPython.display import Image as IPyImage
IPyImage(filename=gif_path)